# submitted: one-GPU replay and stage timing

Attach the official test JPEGs, the public code snapshot and the released weights. No test labels, private caches, credentials or Kaggle API calls are used.

Use an existing compatible CUDA environment or a mounted wheelhouse; inference itself works offline. Edit the three input mounts below. This notebook does not upload a submission.

The child runs the released submit entry point with temporary timing wrappers. Times are exclusive: nested feature work is subtracted from tree and selection time. Tree time includes candidate RLE encoding; RLE/CSV is final serialization. GPU work is synchronized. Total time covers runtime imports through CSV writing, excluding checksum preflight and reference comparison. CPU memory is the OS process high-water RSS; GPU memory reports allocated and reserved peaks.

Replay agreement is against the bundled predictions, **not test ground truth or a competition score**. Masks only match within the same observation. Outputs: submission.csv, inference_timings.csv, replay_summary.csv, run_metadata.json.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Attach the public snapshot, released weights and official competition JPEGs.
CODE = Path("/kaggle/input/REPLACE_CODE_DATASET/filament-segmentation-2026")
WEIGHTS = Path("/kaggle/input/REPLACE_WEIGHT_DATASET")
IMAGES = Path("/kaggle/input/REPLACE_COMPETITION_MOUNT/test_images")
OUT = Path("/kaggle/working/submitted-timing")
WHEELHOUSE = None  # Optional mounted wheel directory for offline installation.

assert (CODE / "tools/timing_replay.py").is_file(), CODE
assert IMAGES.is_dir(), IMAGES
assert not OUT.exists(), "Choose a new output directory."
if WHEELHOUSE is not None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links", str(WHEELHOUSE), "-r", str(CODE / "requirements.txt")],
                   check=True)


In [ ]:
# Restrict the child to one GPU, even on Kaggle's two-T4 runtime.
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
subprocess.run([
    sys.executable, "-m", "tools.timing_replay",
    "--images", str(IMAGES), "--weights-dir", str(WEIGHTS),
    "--out-dir", str(OUT), "--device", "cuda:0", "--tta-batch", "4",
    "--expected-images", "180",
], cwd=CODE, env=env, check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(OUT / "inference_timings.csv"))
display(pd.read_csv(OUT / "replay_summary.csv"))
print((OUT / "run_metadata.json").read_text())
